# Chapter 2 — Modern Liability
### *Modern Strain Gage Technology* — Companion Notebook

*Documented measurement is not bureaucratic overhead — it is evidence that outlives everyone who made the decision.*

Companion notebook to Chapter 2 — Modern Liability.

Engineering decisions made today can be litigated a decade from now. A structure tested in 2025 may be the subject of discovery requests in 2035 — after the engineers who ran the test have changed jobs, the company that built it has been acquired, and the test facility has been reconfigured. What survives that decade? Documented measurement records. What does not? Everything else.

The legal and economic context for measurement is often neglected in engineering education. The focus is on technical accuracy — getting the right number. But the second question — can you prove you got the right number — is equally important in the real world. Calibration certificates, data acquisition configurations, environmental conditions, and raw data archives are not bureaucratic overhead. They are evidence. They allow an engineer to stand behind measurements made years ago in a forum where "I remember it was around 1000 microstrain" is not acceptable.

This notebook visualizes one aspect of that context: the exponential decay of tacit knowledge over time, contrasted with the persistence of documented data. Personnel memories fade, people leave, organizations change — while documented records persist. The gap between the two curves represents the *documentation advantage*: the ability to defend decisions that were made before anyone knew they would need defending.

**This notebook demonstrates:**
1. Tacit knowledge decay vs. documented measurement records — survival curves over a 20-year post-test period, with the litigation window marked

In [ ]:
# Colab setup: install the packages this notebook uses.
# !pip install numpy matplotlib

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

# This notebook is fully deterministic: the curves are analytic decay functions
# with no random sampling, so no RNG seed is required for reproducibility.

OUT_DIR = Path('../src/media/ch02')
OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 — Tacit Knowledge Decay vs. Documented Measurement Records

The most reliable form of institutional memory is documentation. Personnel leave. Memories fade. Organizations reorganize. But a properly archived data file — with the acquisition configuration, calibration certificates, and environmental conditions recorded — is as legible in year fifteen as it was in year one, assuming it was stored in a non-proprietary format and the storage medium remains readable.

The exponential decay model used here is a simplification, but it captures the qualitative relationship correctly. Tacit knowledge — what the engineer who ran the test remembers — decays roughly exponentially with time as personnel retire or change roles. Documented knowledge is more persistent, subject mainly to media degradation and format obsolescence risks that accumulate only after many years. The slight downward slope in the documented-records curve after year fifteen reflects this format risk: test data stored in a proprietary binary format from a discontinued acquisition system becomes increasingly difficult to read.

The litigation window annotated at five to ten years post-test represents the typical range within which post-test legal proceedings are likely to require engineering testimony. Within that window, the documentation advantage is already decisive: tacit knowledge has decayed to roughly 20–40% of its original level, while documented records remain near their original completeness. The gap between the curves — the documentation advantage — is the quantitative argument for rigorous archiving practice.

In [ ]:
# --- Model parameters (decay rates in 1/year) ---
YEARS_SPAN = 20               # length of the post-test period we model (years)
N_POINTS = 200               # resolution of the time axis
TACIT_DECAY_RATE = 0.30      # exponential decay of personnel memory (1/yr)
DOC_BASELINE = 0.95          # documented records: near-complete retention (fraction)
DOC_FORMAT_RISK_RATE = 0.05  # slow decay once format/media obsolescence sets in (1/yr)
DOC_RISK_ONSET_YEAR = 15     # year at which format/media risk begins to bite
LITIGATION_WINDOW = (5, 10)  # typical years-after-test for post-test legal proceedings


def retention_curves(years_span=YEARS_SPAN, n_points=N_POINTS):
    """Return (t, tacit, documented) knowledge-retention fractions over time.

    tacit      -- undocumented personnel memory, decaying exponentially as
                  staff retire or change roles.
    documented -- archived measurement records, held near DOC_BASELINE until
                  format/media obsolescence begins at DOC_RISK_ONSET_YEAR,
                  after which it decays slowly.
    All values are fractions in [0, 1]; multiply by 100 to read as percent.
    """
    t = np.linspace(0, years_span, n_points)
    tacit = np.exp(-TACIT_DECAY_RATE * t)
    documented = np.full_like(t, DOC_BASELINE)
    late = t > DOC_RISK_ONSET_YEAR
    documented[late] *= np.exp(-DOC_FORMAT_RISK_RATE * (t[late] - DOC_RISK_ONSET_YEAR))
    return t, tacit, documented


t, tacit, documented = retention_curves()
lit_start, lit_end = LITIGATION_WINDOW

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t, tacit * 100, 'tomato', linewidth=2, label='Tacit knowledge (personnel memory)')
ax.plot(t, documented * 100, 'steelblue', linewidth=2, label='Documented measurement records')
ax.axvline(lit_start, color='gray', linestyle='--', linewidth=0.8,
           label=f'Typical litigation window ({lit_start}–{lit_end} yr)')
ax.axvline(lit_end, color='gray', linestyle='--', linewidth=0.8)
ax.fill_between(t, tacit * 100, documented * 100, alpha=0.1, color='steelblue',
                label='Documentation advantage')
ax.set_xlabel('Years after original test')
ax.set_ylabel('Knowledge retained (%)')
ax.set_title('Tacit Knowledge Decay vs. Documented Measurement Records')
ax.legend(fontsize=9)
ax.grid(True, linestyle=':', alpha=0.5)
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig02_nb1_documentation_value_time.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Where to take this next

The model above is deliberately simple — a pair of decay curves meant to make one argument visible. Here are concrete ways to push it further:

1. **Calibrate the decay rates to real data.** The `TACIT_DECAY_RATE` of 0.30/yr is a plausible guess, not a measurement. Replace it with an estimate drawn from your own organization: plot staff-turnover records or the fraction of a project team still reachable versus years elapsed, fit an exponential, and see how the documentation advantage shifts.

2. **Model format obsolescence explicitly.** Right now documented records decay as one smooth exponential after year 15. Swap in a step-wise *survival* model instead: assign each archive format (open CSV/HDF5 vs. a proprietary binary from a discontinued acquisition system) its own onset year and hazard rate, then compare the readable-fraction curves. This turns "store it in a non-proprietary format" from advice into a quantified difference.

3. **Add a cost or risk axis.** Overlay the expected cost of litigation exposure — probability of a legal event in each year multiplied by the cost of being unable to defend the measurement — and integrate the gap between the curves. That converts the qualitative "documentation advantage" into an expected-value case for an archiving budget.